In [ ]:
!pip install pennylane --quiet

import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.metrics import mean_absolute_error
from skimage.transform import resize

###=== 1. Original oracle: Gray-code traversal + C^nP gates with dyadic powers ===###
def qpipe_algorithm(flat_phases_1, flat_phases_2, img_wires, est_wire, power):
    n = len(img_wires)
    total_states = 2**n
    gray_seq = [i ^ (i >> 1) for i in range(total_states)]

    for w in img_wires:                                   # |0...0> -> |1...1>
        qml.PauliX(wires=w)
    target_wire = img_wires[-1]
    control_wires = [est_wire] + list(img_wires[:-1])

    for step in range(total_states):
        pixel_idx = gray_seq[step]
        if pixel_idx < len(flat_phases_1):
            for phi in (flat_phases_1[pixel_idx], flat_phases_2[pixel_idx]):
                if abs(phi) > 1e-5:
                    qml.ctrl(qml.PhaseShift, control=control_wires)(phi * power, wires=target_wire)
        if step < total_states - 1:                       # one X per Gray transition
            bit = int(np.log2(pixel_idx ^ gray_seq[step + 1]))
            qml.PauliX(wires=img_wires[n - 1 - bit])

    for i, bit in enumerate(format(gray_seq[-1], f'0{n}b')):   # uncompute
        if bit == '0':
            qml.PauliX(wires=img_wires[i])

###=== 2. UCR oracle: uniformly controlled Rz rotations + one diagonal ===###
def ucrz(alpha, controls, target):
    """Uniformly controlled Rz (Mottonen et al. 2004): 2^m Rz and 2^m CNOTs in a Gray-code parity network."""
    m = len(controls)
    if m == 0:
        qml.RZ(alpha[0], wires=target)
        return
    N = 2**m
    gray = [i ^ (i >> 1) for i in range(N)]
    sign = np.array([[(-1) ** bin(x & g).count('1') for g in gray] for x in range(N)])
    theta = sign.T @ np.asarray(alpha) / N                # solves alpha_x = sum_i (-1)^(x.g_i) theta_i
    for i in range(N):
        qml.RZ(theta[i], wires=target)
        bit = int(np.log2(gray[i] ^ gray[(i + 1) % N]))
        qml.CNOT(wires=[controls[m - 1 - bit], target])

def diagonal(lam, wires):
    """diag(e^{i lam_x}) up to a global phase, built recursively from UCRz gates (2^n - 2 CNOTs)."""
    if len(wires) == 0:
        return
    lam = np.asarray(lam, dtype=float)
    ucrz(lam[1::2] - lam[0::2], wires[:-1], wires[-1])
    diagonal(0.5 * (lam[0::2] + lam[1::2]), wires[:-1])

def qpipe_algorithm_ucr(flat_phases_1, flat_phases_2, img_wires, est_wires):
    """C-U^(2^k) = UCRz_{e_k}(2^k phi) (I x D_k), paper eq. (28); the q diagonals D_k merge into one."""
    n, q = len(img_wires), len(est_wires)
    phi = np.zeros(2**n)
    phi[:len(flat_phases_1)] = np.asarray(flat_phases_1) + np.asarray(flat_phases_2)   # U_A U_B^dagger
    for k, e in enumerate(reversed(est_wires)):           # last estimation wire carries 2^0
        ucrz((2**k) * phi, img_wires, e)
    diagonal(0.5 * (2**q - 1) * phi, img_wires)

###=== 3. Q-PIPE circuit (same structure for both oracles) ===###
def build_qpipe(n_est, n_pos, oracle="ucr", shots=None):
    est_wires = list(range(n_est))
    img_wires = list(range(n_est, n_est + n_pos))

    @qml.qnode(qml.device('default.qubit', wires=n_est + n_pos))
    def circuit(flat_p1, flat_p2):
        for w in est_wires + img_wires:
            qml.Hadamard(wires=w)
        if oracle == "cnp":
            for i, est_wire in enumerate(reversed(est_wires)):
                qpipe_algorithm(flat_p1, flat_p2, img_wires, est_wire, 2**i)
        else:
            qpipe_algorithm_ucr(flat_p1, flat_p2, img_wires, est_wires)
        qml.adjoint(qml.QFT)(wires=est_wires)
        return qml.probs(wires=est_wires + img_wires)

    return qml.set_shots(circuit, shots=shots) if shots else circuit

###=== 4. Circuit diagrams: 2x2 image and its horizontal shift, 3 estimation qubits ===###
dummy_image = np.array([[10, 255],
                        [128, 64]])
dummy_shift = np.roll(dummy_image, 1, axis=1); dummy_shift[:, 0] = 0
dummy_p1 = (dummy_image.flatten() / 255.0) * np.pi        # half-spectrum mapping [0, pi]
dummy_p2 = -(dummy_shift.flatten() / 255.0) * np.pi       # negative phases -> subtraction

for oracle, title in [("cnp", "Q-PIPE oracle: Gray code + C^nP"), ("ucr", "Q-PIPE oracle: UCR synthesis")]:
    fig, ax = qml.draw_mpl(build_qpipe(3, 2, oracle), decimals=2)(dummy_p1, dummy_p2)
    fig.suptitle(title)
    plt.show()

###=== 5. Verification: both oracles implement the same unitary; CNOT counts ===###
U_cnp = qml.matrix(build_qpipe(3, 2, "cnp"))(dummy_p1, dummy_p2)
U_ucr = qml.matrix(build_qpipe(3, 2, "ucr"))(dummy_p1, dummy_p2)
g = np.vdot(U_ucr.ravel(), U_cnp.ravel()); g /= abs(g)        # global phase
print(f"max |U_cnp - e^(i g) U_ucr| = {np.abs(U_cnp - g * U_ucr).max():.2e}")

GATE_SET = {"CNOT", "RZ", "RX", "RY", "PhaseShift", "Hadamard", "PauliX", "GlobalPhase"}
def cnot_count(circuit, *args):
    tape = qml.workflow.construct_tape(circuit)(*args)
    [tape], _ = qml.transforms.decompose(tape, gate_set=GATE_SET)
    return sum(op.name == "CNOT" for op in tape.operations)

q, n = 8, 4                                                # paper, Appendix B: 4x4 image, 8 estimation qubits
rng = np.random.default_rng(0)                             # worst case: all pixels nonzero
p1, p2 = rng.uniform(0.1, np.pi, 2**n), -rng.uniform(0.1, np.pi, 2**n)
N = 2**n
print(f"CNOTs, C^nP: {cnot_count(build_qpipe(q, n, 'cnp'), p1, p2)}")
print(f"CNOTs, UCR : {cnot_count(build_qpipe(q, n, 'ucr'), p1, p2)} "
      f"(expected (q+1)N-2 + QFT = {(q + 1) * N - 2} + {q * (q - 1) + 3 * (q // 2)})")

In [ ]:
###=== 6. Application: horizontal QED on the first five MNIST images ===###
N_EST = 8          # estimation qubits (q)
SIZE = 4           # 4 -> paper Appendix B (fetch_openml 28x28 -> 4x4); 8 -> 8x8 images
P_TH = 5e-4        # read-out threshold, P_th <= C_D / 2^n
SHOTS = None       # None = exact probabilities; 4096 = sampled read-out as in the paper
COMPARE_CNP = True # also run the C^nP oracle and check that both distributions coincide

n_pos = int(np.ceil(np.log2(SIZE * SIZE)))
X = fetch_openml('mnist_784', version=1, cache=True, parser='auto').data.values.reshape(-1, 28, 28)
images = []
for idx in range(5):
    im = resize(X[idx], (SIZE, SIZE), anti_aliasing=True, preserve_range=True)
    images.append(im / np.max(im) * 255.0)                # peak normalization to [0, 255]

def qed_phases(img):
    """Half-spectrum phases for horizontal QED, I(x,-1) = 0 at the boundary."""
    shifted = np.roll(img, shift=1, axis=1); shifted[:, 0] = 0
    return (img.flatten() / 255.0) * np.pi, -(shifted.flatten() / 255.0) * np.pi, shifted, np.abs(img - shifted)

def extract_magnitude(probs, shape, n_est, n_pos, prob_threshold=P_TH):
    """Probability-weighted read-out |dI| = 2 I_max min(f, 1-f), f = k / 2^q (paper, eq. 16)."""
    P = np.asarray(probs).reshape(2**n_est, 2**n_pos)[:, :np.prod(shape)]   # P[k, x]
    f = np.arange(2**n_est) / 2**n_est
    W = np.where(P > prob_threshold, P, 0.0)
    den = W.sum(axis=0)
    return ((2 * 255.0 * np.minimum(f, 1 - f)) @ W / np.where(den == 0, 1.0, den)).reshape(shape)

circuit_ucr = build_qpipe(N_EST, n_pos, "ucr", SHOTS)
circuit_cnp = build_qpipe(N_EST, n_pos, "cnp", SHOTS)

fig, axes = plt.subplots(5, 4, figsize=(12, 15))
for idx, img in enumerate(images):
    p1, p2, shifted, grad = qed_phases(img)
    probs = circuit_ucr(p1, p2)
    q_grad = extract_magnitude(probs, img.shape, N_EST, n_pos)
    mae = mean_absolute_error(grad.flatten(), q_grad.flatten())
    msg = f"Image {idx + 1}: MAE (UCR) = {mae:.3f}"
    if COMPARE_CNP and SHOTS is None:
        msg += f" | max|P_UCR - P_CnP| = {np.abs(probs - circuit_cnp(p1, p2)).max():.1e}"
    print(msg)

    for ax, im, title in zip(axes[idx], [img, shifted, grad, q_grad],
                             ["Original", "Displaced", "Classical gradient", f"Q-PIPE (UCR)\nMAE: {mae:.2f}"]):
        ax.imshow(np.clip(im, 0, 255), cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')

plt.tight_layout()
plt.savefig(f"qpipe_ucr_qed_mnist_{SIZE}x{SIZE}.png", dpi=300)
plt.show()